# The $\alpha_s$ method

Very similar to the t-plot method, the $\alpha_s$ method compares an isotherm on
a porous material with one that was taken on a reference non-porous surface. The
reference isotherm should be measured over a wide pressure range, to be able to
compare loading values. First, make sure the data is imported notebook.

In [2]:
# import isotherms
%run import.ipynb

# import the characterisation module
import pygaps.characterisation as pgc

Selected 5 isotherms with nitrogen at 77K
Selected 2 room temperature calorimetry isotherms
Selected 2 isotherms for IAST calculation
Selected 3 isotherms for isosteric enthalpy calculation
There is no AIF version specified in the file. The parser will still attempt to read it using the latest schema (v1.0).


File declares AIF version v1.0 (AIF v1.0) but is missing required tag(s): _adsnt_material_name, _exptl_method. Parsing will continue on a best-effort basis.


Could not parse parameter material_mass, currently unknown


The silica isotherm is used as a practical non-porous reference for MCM-41. First, convert both data sets to relative pressure and compare them only over their measured overlap:

In [3]:
import pygaps

iso_1 = next(i for i in isotherms_n2_77k if i.material == 'MCM-41')
iso_2 = next(i for i in isotherms_n2_77k if i.material == 'SiO2')

# Put both data sets on the same pressure basis and restrict MCM-41 to their overlap.
iso_2.convert_pressure(mode_to='relative')
target_pressure = iso_1.pressure(branch='ads', pressure_mode='relative')
reference_pressure = iso_2.pressure(branch='ads', pressure_mode='relative')
common_min = max(target_pressure.min(), reference_pressure.min())
common_max = min(target_pressure.max(), reference_pressure.max())

isotherm_data = iso_1.data(branch='ads').copy()
isotherm_data['relative_pressure'] = target_pressure
isotherm_data = isotherm_data.loc[
    isotherm_data['relative_pressure'].between(common_min, common_max)
].drop(columns='relative_pressure')
iso_1 = pygaps.PointIsotherm.from_isotherm(
    iso_1,
    isotherm_data=isotherm_data,
    pressure_key=iso_1.pressure_key,
    loading_key=iso_1.loading_key,
)

print(iso_1.material)
print(iso_2.material)
results = pgc.alpha_s(iso_1, reference_isotherm=iso_2, verbose=True)

MCM-41
SiO2
For linear region 0


The slope is 3.627 and the intercept is 1.927, with a correlation coefficient of 0.9996


The adsorbed volume is 0.067 cm3/g and the area is 230.5 m2/g


For linear region 1


The slope is 0.488 and the intercept is 8.842, with a correlation coefficient of 0.9729


The adsorbed volume is 0.307 cm3/g and the area is 31.02 m2/g


The first calculation uses the overlapping measured pressure range only. We can also fit a BET model to the reference data and compare the resulting alpha-s curve.

In [4]:
import pygaps

model = pygaps.ModelIsotherm.from_pointisotherm(iso_2, model='BET', verbose=True)

Attempting to model using BET.


Model BET success, RMSE is 0.0311


With the reference model fitted, we can repeat the alpha-s calculation without extrapolating beyond the measured target range.

In [5]:
results = pgc.alpha_s(iso_1, model, verbose=True)

For linear region 0


The slope is 5.818 and the intercept is 0.1125, with a correlation coefficient of 1


The adsorbed volume is 0.00391 cm3/g and the area is 352.2 m2/g


For linear region 1


The slope is 0.3735 and the intercept is 8.895, with a correlation coefficient of 0.9903


The adsorbed volume is 0.309 cm3/g and the area is 22.61 m2/g


The results don't look that bad, considering all our assumptions and modelling. 
There are other parameters which can be specified for the $\alpha_s$ function such as:

- The relative pressure to use as the reducing pressure
- The known area of the reference material. If this is not specified, the BET method is used to calculate the surface area.
- As in the t-plot function, the limits for the straight line selection.

Read more about the theory in the
[documentation of the module](../reference/characterisation/alphas_plot.rst) and
in the [manual](../manual/characterisation.rst).